# Clasificación de perros y gatos con una CNN
**Objetivo:** clasificar imágenes como gato o perro usando Cats vs Dogs.

Este notebook cubre exploración, preprocesamiento, particiones, aumento de datos, construcción y entrenamiento de una CNN, métricas, matriz de confusión, al menos 10 predicciones visuales y guardado del modelo. Está pensado para ejecutarse en Google Colab.

## 1. Instalación e importaciones

In [ ]:
!pip -q install tensorflow-datasets scikit-learn seaborn
import os, json, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report, confusion_matrix
print("TensorFlow:", tf.__version__)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. Dataset y descripción

In [ ]:
(ds_train, ds_val, ds_test), info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    as_supervised=True, with_info=True, shuffle_files=True
)
print(info)
print("Clases:", info.features["label"].names)

Cats vs Dogs contiene imágenes de gatos y perros con etiquetas. Se usa una división 80/10/10 para entrenamiento, validación y prueba. La división de prueba se reserva para la evaluación final.

## 3. Exploración y visualización de imágenes

In [ ]:
label_names = info.features["label"].names
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, (image, label) in zip(axes.flat, ds_train.take(10)):
    ax.imshow(image.numpy())
    ax.set_title(label_names[int(label)])
    ax.axis("off")
plt.suptitle("Muestra de imágenes del dataset")
plt.tight_layout()
plt.show()

In [ ]:
# Conteo de etiquetas en una muestra del conjunto de entrenamiento
counts = {name: 0 for name in label_names}
for _, label in ds_train.take(5000):
    counts[label_names[int(label)]] += 1
plt.bar(counts.keys(), counts.values())
plt.title("Distribución de clases en una muestra de 5.000 ejemplos")
plt.ylabel("Cantidad")
plt.show()
print(counts)

## 4. Preprocesamiento y normalización

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.int32)

train_ds = (ds_train.map(preprocess, num_parallel_calls=AUTOTUNE)
            .shuffle(2000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds = (ds_val.map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE).prefetch(AUTOTUNE))
test_ds = (ds_test.map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE).prefetch(AUTOTUNE))

La normalización de píxeles se aplica dentro del modelo con `Rescaling(1./255)`, para asegurar que entrenamiento e inferencia usen exactamente el mismo procesamiento.

## 5. Construcción de la CNN y Data Augmentation

In [ ]:
from tensorflow.keras import layers, models

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

inputs = layers.Input(shape=(160, 160, 3))
x = augmentation(inputs)
x = layers.Rescaling(1./255)(x)
for filters in (32, 64, 128):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(2, activation="softmax")(x)
model = models.Model(inputs, outputs, name="cnn_gatos_perros")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Entrenamiento

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("models/cats_dogs_cnn.keras",
        monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
        patience=2, min_lr=1e-6, verbose=1)
]
history = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=callbacks)
model.save("models/cats_dogs_cnn.keras")
with open("reports_history.json", "w") as f:
    json.dump({k:[float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

## 7. Gráficas del entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Entrenamiento")
axes[0].plot(history.history["val_accuracy"], label="Validación")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Época"); axes[0].legend()
axes[1].plot(history.history["loss"], label="Entrenamiento")
axes[1].plot(history.history["val_loss"], label="Validación")
axes[1].set_title("Loss"); axes[1].set_xlabel("Época"); axes[1].legend()
plt.tight_layout()
plt.savefig("reports/curvas_entrenamiento.png", dpi=150)
plt.show()

## 8. Evaluación en el conjunto de prueba

In [ ]:
y_true, y_pred, y_prob = [], [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    y_prob.extend(probs.tolist())

print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicción"); plt.ylabel("Etiqueta real")
plt.title("Matriz de confusión")
plt.tight_layout()
plt.savefig("reports/matriz_confusion.png", dpi=150)
plt.show()

## 9. Al menos 10 predicciones con imágenes

In [ ]:
# Visualiza 10 ejemplos del conjunto de prueba con etiqueta real, predicción y confianza.
examples = list(ds_test.take(10))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, (image, label) in zip(axes.flat, examples):
    resized = tf.image.resize(image, IMG_SIZE)
    probs = model.predict(tf.expand_dims(tf.cast(resized, tf.float32), 0), verbose=0)[0]
    pred = int(np.argmax(probs))
    ax.imshow(image.numpy())
    ax.set_title(f"Real: {label_names[int(label)]}\nPred: {label_names[pred]} ({probs[pred]:.1%})")
    ax.axis("off")
plt.tight_layout()
plt.savefig("reports/10_predicciones.png", dpi=150)
plt.show()

## 10. Guardado y descarga de artefactos

In [ ]:
model.save("models/cats_dogs_cnn.keras")
print("Modelo guardado:", os.path.abspath("models/cats_dogs_cnn.keras"))
print("Gráficas generadas en la carpeta reports/")

## 11. Usar el modelo en Streamlit
Una vez descargado o copiado el modelo a `models/cats_dogs_cnn.keras`, desde la raíz del repositorio ejecuta:

```bash
pip install -r requirements.txt
streamlit run app/streamlit_app.py
```

**Nota:** los resultados numéricos aparecen al ejecutar las celdas. No se incluyen métricas inventadas: deben reportarse los valores que produzca el entrenamiento real.